<a href="https://colab.research.google.com/github/aleckalkahmudyanadzo-cyber/skin_lesion_triage_1/blob/main/skin_lesion_triage/colab/train_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Skin Lesion Triage — Colab Training Notebook

MSU Final Year Project — Aleck Mudyanadzo (R252845M) & Primrose S. Ncube (R253036M)

**Before you start:**
1. Runtime -> Change runtime type -> GPU (T4 is fine).
2. Have your `skin_lesion_triage.zip` project folder ready to upload.
3. Have your Kaggle API token (`kaggle.json`) ready — get it from kaggle.com/settings -> API -> Create New Token.

This notebook: uploads your project -> downloads HAM10000 -> preps the data -> trains MobileNetV2 and ResNet50 -> lets you download the trained `.keras` files to drop into your local `models/` folder.

## 1. Upload your project zip

In [1]:
from google.colab import files
print('Select your skin_lesion_triage.zip file...')
uploaded = files.upload()

Select your skin_lesion_triage.zip file...


Saving skin_lesion_triage.zip to skin_lesion_triage.zip


In [3]:
import zipfile, glob, os

zip_name = [f for f in uploaded.keys() if f.endswith('.zip')][0]
with zipfile.ZipFile(zip_name, 'r') as z:
    for info in z.infolist():
        info.filename = info.filename.replace('\\', '/')
        z.extract(info, '/content/')

%cd /content/skin_lesion_triage
!ls

/content/skin_lesion_triage
app    config.py  instance  __pycache__  requirements.txt  scripts
colab  data	  models    README.md	 run.py		   setup.bat


## 2. Install dependencies

In [2]:
!pip install -q kaggle opencv-python-headless
# TensorFlow, numpy, pandas, scikit-learn already come preinstalled on Colab

## 3. Upload your Kaggle API token

In [9]:
print('Select your kaggle.json file...')
kaggle_upload = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle token installed.')

Select your kaggle.json file...


Saving kaggle.json to kaggle.json
Kaggle token installed.


## 4. Download and prepare HAM10000

In [4]:
!python scripts/prepare_data.py --kaggle_download

[*] Downloading kmader/skin-cancer-mnist-ham10000 into /content/skin_lesion_triage/data/raw ...
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:50<00:00, 111MB/s]

[*] Download complete.
[*] Using metadata: /content/skin_lesion_triage/data/raw/HAM10000_metadata.csv
[*] Class distribution:
binary_label
benign       8061
malignant    1954
Name: count, dtype: int64
[*] Indexed 10015 images across 4 folders.
[*] train: copied 7010 images (0 missing/skipped)
[*] val: copied 1502 images (0 missing/skipped)
[*] test: copied 1503 images (0 missing/skipped)

[✓] Data preparation complete.
    Output at: /content/skin_lesion_triage/data/processed


In [9]:
import shutil, os
src = '/content/drive/MyDrive/skin_lesion_triage_models'
dest = 'models'
os.makedirs(dest, exist_ok=True)

if os.path.exists(src):
    for fname in os.listdir(src):
        if fname.endswith('.keras'):
            shutil.copy(os.path.join(src, fname), dest)
            print('Restored', fname)
else:
    print(f"Warning: Source directory '{src}' not found. No models to restore.")

In [14]:
%%writefile scripts/resume_threshold_tuning.py
"""
resume_threshold_tuning.py

Use this after a Colab runtime reset wiped the *_threshold.json files but
the trained .keras model weights survived (e.g. were already copied to
Google Drive). This does NOT retrain anything, it just reloads a saved
model, re-tunes its decision threshold against the validation set, and
reports both validation and test set numbers, so you get back the real
Objective 2 figures without paying for another 25-epoch training run.

Requires data/processed/{train,val,test} to exist (rerun prepare_data.py
first if that is also gone) and the .keras files to be present in models/
(copy them back down from Drive first if needed).

USAGE (Colab, from the skin_lesion_triage folder):
    !python scripts/resume_threshold_tuning.py --arch mobilenetv2
    !python scripts/resume_threshold_tuning.py --arch resnet50
"""

import argparse
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.metrics import recall_score, precision_score

from train_model import build_datasets, tune_threshold, MODEL_DIR, ARCH_BUILDERS


def evaluate_at_threshold(model, ds, threshold: float) -> dict:
    y_true = np.concatenate([y.numpy() for _, y in ds], axis=0).ravel()
    y_prob = model.predict(ds, verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "recall": round(float(recall_score(y_true, y_pred)), 4),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--arch", choices=list(ARCH_BUILDERS.keys()), required=True)
    args = parser.parse_args()

    _, model_filename = ARCH_BUILDERS[args.arch]
    model_path = MODEL_DIR / f"{model_filename}.keras"
    if not model_path.exists():
        raise FileNotFoundError(
            f"{model_path} not found. Copy it back from Google Drive into "
            f"the models/ folder first."
        )

    print(f"[*] Loading {model_path}")
    model = tf.keras.models.load_model(model_path)

    print("[*] Rebuilding datasets from data/processed ...")
    train_ds, val_ds, test_ds, class_names = build_datasets()
    print(f"[*] Class order (0/1): {class_names}")

    print("\n" + "=" * 60)
    print(f"TUNING DECISION THRESHOLD for {args.arch} on validation set")
    print("=" * 60)
    threshold_info = tune_threshold(model, val_ds, target_recall=0.85)
    threshold_path = MODEL_DIR / f"{model_filename}_threshold.json"
    with open(threshold_path, "w") as f:
        json.dump(threshold_info, f, indent=2)

    if threshold_info["met_target"]:
        print(f"[✓] Threshold {threshold_info['threshold']} reaches the 0.85 recall "
              f"target (val recall={threshold_info['val_recall_at_threshold']}, "
              f"val precision={threshold_info['val_precision_at_threshold']}).")
    else:
        print(f"[!] No threshold reached 0.85 recall on validation. Using best "
              f"available: {threshold_info['threshold']} "
              f"(val recall={threshold_info['val_recall_at_threshold']}).")
    print(f"    Saved to {threshold_path}")

    print("\n" + "=" * 60)
    print(f"TEST SET performance at tuned threshold ({threshold_info['threshold']})")
    print("=" * 60)
    test_results = evaluate_at_threshold(model, test_ds, threshold_info["threshold"])
    print(f"    Test recall:    {test_results['recall']}")
    print(f"    Test precision: {test_results['precision']}")

    if test_results["recall"] >= 0.85:
        print("\n[✓] Objective 2 target met on the held-out test set.")
    else:
        print("\n[!] Test recall below 0.85 -- report this honestly in Chapter 5, "
              "do not round up. Worth discussing as a limitation if it happens.")


if __name__ == "__main__":
    main()

Overwriting scripts/resume_threshold_tuning.py


In [15]:
!python scripts/resume_threshold_tuning.py --arch mobilenetv2


2026-09-24 07:15:09.847676: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/content/skin_lesion_triage/scripts/resume_threshold_tuning.py", line 94, in <module>
    main()
    ~~~~^^
  File "/content/skin_lesion_triage/scripts/resume_threshold_tuning.py", line 49, in main
    raise FileNotFoundError(
    ...<2 lines>...
    )
FileNotFoundError: /content/skin_lesion_triage/models/mobilenetv2_skin_lesion.keras not found. Copy it back from Google Drive into the models/ folder first.


## 5. Train MobileNetV2 (fast, deploy-friendly)
Target: validation recall >= 0.85 on the malignant class (Objective 2).

In [11]:
%cd scripts
!python train_model.py --arch mobilenetv2 --epochs_head 10 --epochs_finetune 15
%cd ..

/content/skin_lesion_triage/scripts
2026-09-24 03:33:40.399900: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 7010 files belonging to 2 classes.
2026-09-24 03:33:48.471538: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1790220828.473029    5063 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Found 1502 files belonging to 2 classes.
Found 1503 files belonging to 2 classes.
[*] Class order (0/1): ['benign', 'malignant']
[*] Class counts -> benign: 5642, mal

In [5]:
!pwd
!ls /content

/
sample_data


## 6. Train ResNet50 (for comparison, per Objective 2)

In [12]:
%cd scripts
!python train_model.py --arch resnet50 --epochs_head 10 --epochs_finetune 15
%cd ..

/content/skin_lesion_triage/scripts
2026-09-24 04:14:22.361939: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 7010 files belonging to 2 classes.
2026-09-24 04:14:27.555633: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1790223267.557238   16019 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Found 1502 files belonging to 2 classes.
Found 1503 files belonging to 2 classes.
[*] Class order (0/1): ['benign', 'malignant']
[*] Class counts -> benign: 5642, mal

## 7. Download the trained models
Drop both `.keras` files into your local project's `models/` folder.

In [15]:
from google.colab import files
import os

for fname in os.listdir('models'):
    if fname.endswith('.keras'):
        print('Downloading', fname)
        files.download(os.path.join('models', fname))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. (Optional) Save models straight to Google Drive instead
Useful if the direct download above times out for large files.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dest = '/content/drive/MyDrive/skin_lesion_triage_models'
os.makedirs(dest, exist_ok=True)
for fname in os.listdir('models'):
    if fname.endswith('.keras'):
        shutil.copy(os.path.join('models', fname), dest)
        print('Copied', fname, 'to Drive:', dest)

Mounted at /content/drive
Copied resnet50_skin_lesion.keras to Drive: /content/drive/MyDrive/skin_lesion_triage_models
Copied mobilenetv2_skin_lesion.keras to Drive: /content/drive/MyDrive/skin_lesion_triage_models


In [6]:
import shutil, os
src = '/content/skin_lesion_triage/models'
dest = '/content/drive/MyDrive/skin_lesion_triage_models'
os.makedirs(dest, exist_ok=True)

if not os.path.exists(src):
    print(f"Error: Source directory '{src}' not found. Please ensure the project zip is uploaded and models are trained.")
else:
    for fname in os.listdir(src):
        if fname.endswith('.json'):
            shutil.copy(os.path.join(src, fname), dest)
            print('Copied', fname)

Error: Source directory '/content/skin_lesion_triage/models' not found. Please ensure the project zip is uploaded and models are trained.


In [3]:
!ls -la /content/skin_lesion_triage/models

ls: cannot access '/content/skin_lesion_triage/models': No such file or directory


In [4]:
!ls /content
!ls /content/drive/MyDrive/skin_lesion_triage_models

drive  sample_data


In [7]:
!ls /content/drive/MyDrive/skin_lesion_triage_models